In [ ]:
import pandas as pd
import chromadb
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

# =====================================================================
# STEP 1: TRAIN THE INTENT ROUTER (CLASSIFIER)
# =====================================================================
print("--- Step 1: Training the Intent Router ---")

# Let's mock a subset of labeled intents based on your domain datasets
# This tells the agent whether to process a banking complaint or look up information.
training_data = {
    "text": [
        # Banking / Complaint Intents
        "How do I activate my new credit card?",
        "My debit card is lost, please help",
        "Why was I charged twice for this transaction?",
        "I need to report a fraudulent charge on my account",
        "The exchange rate for my cash withdrawal was wrong",
        # General Information / Search Intents
        "What is the capital of France?",
        "Explain how photosynthesis works in plants",
        "Who wrote the play Romeo and Juliet?",
        "How far is the moon from the earth?",
        "Tell me about the history of the internet"
    ],
    "label": [
        "banking_action", "banking_action", "banking_action", "banking_action", "banking_action",
        "general_search", "general_search", "general_search", "general_search", "general_search"
    ]
}

df = pd.DataFrame(training_data)

# Vectorize the text using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["text"])
y = df["label"]

# Train a fast Logistic Regression classifier
router_model = LogisticRegression()
router_model.fit(X, y)
print("Intent Router trained successfully!")


# =====================================================================
# STEP 2: LOAD KNOWLEDGE BASE (CHROMADB VECTORS)
# =====================================================================
print("\n--- Step 2: Loading Knowledge Base (ChromaDB) ---")
# Streaming 500 samples from MS MARCO to act as our general knowledge base
dataset_stream = load_dataset("microsoft/ms_marco", "v2.1", split="train", streaming=True)
sample_data = list(dataset_stream.take(500))

# Initialize Ephemeral Chroma DB
chroma_client = chromadb.EphemeralClient()
collection = chroma_client.create_collection(name="kb_collection")

# Extract and insert passages
documents, ids = [], []
for idx, item in enumerate(sample_data):
    passages = item['passages']['passage_text']
    for p_idx, text in enumerate(passages):
        documents.append(text)
        ids.append(f"doc_{idx}_{p_idx}")

# Batch insert in blocks of 200
batch_size = 200
for i in range(0, len(documents), batch_size):
    collection.add(
        documents=documents[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )
print(f"Indexed {collection.count()} general knowledge passages into ChromaDB.")


# =====================================================================
# STEP 3: THE ROUTING AGENT PIPELINE
# =====================================================================
def run_agent(user_query: str):
    print(f"\n[User Query]: \"{user_query}\"")
    
    # 1. Vectorize query and predict intent category
    query_vector = vectorizer.transform([user_query])
    intent_prediction = router_model.predict(query_vector)[0]
    probabilities = router_model.predict_proba(query_vector)[0]
    confidence = max(probabilities)
    
    print(f"[Agent Brain]: Classified intent as '{intent_prediction}' (Confidence: {confidence:.2%})")
    
    # 2. Route the execution path
    if intent_prediction == "banking_action":
        print("[Agent Path] 💳 Routing to Customer Support Service Desk...")
        print(f"-> Initiating secure customer protocol for request: '{user_query}'")
        
    elif intent_prediction == "general_search":
        print("[Agent Path] 🔍 Routing to Local Vector Database (RAG Search)...")
        # Query ChromaDB for top matching passage
        results = collection.query(query_texts=[user_query], n_results=1)
        
        if results['documents'] and len(results['documents'][0]) > 0:
            print(f"-> Retrieved Context: \"{results['documents'][0][0]}\"")
        else:
            print("-> No matching documents found in database.")


# =====================================================================
# STEP 4: TEST RUNNING THE AGENT
# =====================================================================
print("\n--- Step 4: Testing Agent Routing Capability ---")

# Test Case A: Should route to banking actions
run_agent("Can you help me unlock my debit card?")

# Test Case B: Should route to RAG database retrieval
run_agent("Tell me about the capital of France")

--- Step 1: Training the Intent Router ---
Intent Router trained successfully!

--- Step 2: Loading Knowledge Base (ChromaDB) ---
